In [11]:
import mesa
import networkx as nx
import numpy as np
from mesa.visualization import SolaraViz, make_space_component
import solara
import matplotlib.pyplot as plt

In [12]:
class NetworkAgent(mesa.Agent):
    def __init__(self, unique_id, model):
        super().__init__(model)
        self.unique_id = unique_id

In [13]:
class NetworkModel(mesa.Model):
    def __init__(self, n_nodes=10):
        super().__init__()
        
        # Create graph with circular layout
        self.graph = nx.Graph()
        self.graph.add_nodes_from(range(n_nodes))
        
        # Add circular positions as node attributes
        for i in range(n_nodes):
            angle = 2 * np.pi * i / n_nodes
            self.graph.nodes[i]['pos'] = (np.cos(angle), np.sin(angle))
        
        # Create Mesa NetworkGrid - THIS IS THE KEY FIX
        self.space = mesa.space.NetworkGrid(self.graph)
        
        # Add agents to nodes
        for i in range(n_nodes):
            agent = NetworkAgent(i, self)
            self.space.place_agent(agent, i)
        
    def step(self):
        # Add random edge
        nodes = list(self.graph.nodes())
        if len(self.graph.edges()) < 15:  # Max edges
            u, v = self.random.choices(nodes, k=2)
            if u != v and not self.graph.has_edge(u, v):
                self.graph.add_edge(u, v)
        
        # Remove random edge
        if len(self.graph.edges()) > 3:  # Min edges
            edge = self.random.choice(list(self.graph.edges()))
            self.graph.remove_edge(*edge)

In [14]:
# Native Mesa network visualization
def agent_portrayal(agent):
    return {
        "size": 20,
        "color": "lightblue",
        "stroke_color": "black",
        "stroke_width": 2,
    }

In [15]:
@solara.component
def CustomNetworkVisualization(model):
    """Custom network visualization component"""
    # This is required to update the visualization when the model steps
    from mesa.visualization.components.matplotlib_components import update_counter
    update_counter.get()  # This triggers re-rendering on model updates
    
    # Initialize figure using Mesa's thread-safe method
    fig = plt.Figure(figsize=(8, 8))
    ax = fig.subplots()
    
    # Get positions from graph
    pos = nx.get_node_attributes(model.graph, 'pos')
    
    # Draw edges
    for edge in model.graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        ax.plot([x0, x1], [y0, y1], 'gray', alpha=0.6, linewidth=2, zorder=1)
    
    # Draw nodes
    node_x = [pos[node][0] for node in model.graph.nodes()]
    node_y = [pos[node][1] for node in model.graph.nodes()]
    
    ax.scatter(node_x, node_y, 
               s=400, c='lightblue', 
               edgecolors='black', linewidths=2,
               zorder=2)
    
    # Add node labels
    for node in model.graph.nodes():
        x, y = pos[node]
        ax.text(x, y, str(node), 
                ha='center', va='center', 
                fontsize=12, fontweight='bold', zorder=3)
    
    # Set up the plot
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title('Dynamic Network Model')
    ax.axis('off')
    
    # Add network statistics
    num_edges = len(model.graph.edges())
    num_nodes = len(model.graph.nodes())
    density = nx.density(model.graph)
    
    stats_text = f"Nodes: {num_nodes}\nEdges: {num_edges}\nDensity: {density:.3f}"
    ax.text(-1.4, 1.2, stats_text, fontsize=10, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))
    
    # This is required to render the visualization
    solara.FigureMatplotlib(fig)

# Create and run the visualization
model = NetworkModel()
page = SolaraViz(
    model,
    components=[CustomNetworkVisualization],
    name="Network Model with Custom Visualization"
)

# To run the visualization:
page

Cannot show ipywidgets in text